# Routing Evaluation & Auto-Training Framework

Offline evaluation + auto-training with **real metric computation** — responses are evaluated against ground truth using actual loss functions, not simulated scores.

**What this notebook covers:**
1. Realistic MMLU-style benchmark with proper ground truth evaluation
2. Baselines (Random, Oracle, AlwaysStrong, AlwaysWeak)
3. Pareto curve sweep + AUROC, CPT, APGR metrics
4. Preference pair generation + production feedback loop
5. Full AutoTrainer cycle: evaluate → augment → retrain → promote

## 1. Setup — Models with Realistic Accuracy

Each mock model responds with the correct answer at a **per-category accuracy rate** — the metric function (`mmlu_match`) then evaluates the response against ground truth. Losses are real, not simulated.

In [ ]:
import numpy as np
import random
import lunar_router as lr
from lunar_router.core.embeddings import PromptEmbedder, MockEmbeddingProvider
from lunar_router.core.clustering import KMeansClusterAssigner
from lunar_router.core.metrics import mmlu_match  # real metric function

# 5 clusters representing different knowledge domains
centroids = np.array([
    [1.0, 0.0],   # cluster 0: STEM
    [0.0, 1.0],   # cluster 1: humanities
    [0.5, 0.5],   # cluster 2: social science
    [-0.5, 0.5],  # cluster 3: professional
    [0.5, -0.5],  # cluster 4: other
])
assigner = KMeansClusterAssigner(centroids)
embedder = PromptEmbedder(MockEmbeddingProvider(dimension=2))

# 4 model profiles — these are per-cluster error rates from "profiling"
profiles = [
    lr.LLMProfile(
        model_id="gpt-4o",
        psi_vector=np.array([0.08, 0.10, 0.07, 0.06, 0.09]),
        cost_per_1k_tokens=0.005,
        num_validation_samples=500,
        cluster_sample_counts=np.array([100, 100, 100, 100, 100]),
    ),
    lr.LLMProfile(
        model_id="gpt-4o-mini",
        psi_vector=np.array([0.20, 0.15, 0.12, 0.25, 0.22]),
        cost_per_1k_tokens=0.00015,
        num_validation_samples=500,
        cluster_sample_counts=np.array([100, 100, 100, 100, 100]),
    ),
    lr.LLMProfile(
        model_id="claude-3.5-sonnet",
        psi_vector=np.array([0.09, 0.05, 0.11, 0.04, 0.12]),
        cost_per_1k_tokens=0.003,
        num_validation_samples=500,
        cluster_sample_counts=np.array([100, 100, 100, 100, 100]),
    ),
    lr.LLMProfile(
        model_id="llama-3.1-8b",
        psi_vector=np.array([0.35, 0.30, 0.25, 0.40, 0.32]),
        cost_per_1k_tokens=0.00005,
        num_validation_samples=500,
        cluster_sample_counts=np.array([100, 100, 100, 100, 100]),
    ),
]

registry = lr.LLMRegistry()
for p in profiles:
    registry.register(p)

print("Model profiles:")
for p in profiles:
    print(f"  {p.model_id:20s}  accuracy={p.overall_accuracy:.1%}  cost=${p.cost_per_1k_tokens}/1k")

## 2. Create MMLU-Style Dataset & Populate Cache with Real Metrics

Each prompt is a multiple-choice question (A/B/C/D). Models respond with a letter. The `mmlu_match` metric extracts the letter and compares to ground truth. **Loss = 0.0 means correct, 1.0 means wrong — computed from actual string comparison.**

In [ ]:
random.seed(42)
np.random.seed(42)

categories = ["stem", "humanities", "social_science", "professional", "other"]
cat_to_cluster = {c: i for i, c in enumerate(categories)}
answer_choices = ["A", "B", "C", "D"]

# Generate 200 MMLU-style questions with known correct answers
samples = []
for i in range(200):
    cat = categories[i % len(categories)]
    correct = answer_choices[i % 4]
    prompt = (
        f"Question {i} ({cat}): What is the correct answer?\n"
        f"A) Option A\nB) Option B\nC) Option C\nD) Option D\nAnswer:"
    )
    samples.append(lr.PromptSample(prompt=prompt, ground_truth=correct, category=cat))

dataset = lr.PromptDataset(samples, name="mmlu-synthetic")

# Populate cache — each model ACTUALLY responds with a letter,
# and mmlu_match() computes the real loss
cache = lr.ResponseCache()

# Per-model, per-category accuracy rates (realistic numbers)
accuracy_rates = {
    "gpt-4o":           {"stem": 0.90, "humanities": 0.88, "social_science": 0.92, "professional": 0.93, "other": 0.89},
    "gpt-4o-mini":      {"stem": 0.78, "humanities": 0.82, "social_science": 0.85, "professional": 0.72, "other": 0.76},
    "claude-3.5-sonnet": {"stem": 0.89, "humanities": 0.94, "social_science": 0.87, "professional": 0.95, "other": 0.86},
    "llama-3.1-8b":     {"stem": 0.62, "humanities": 0.68, "social_science": 0.72, "professional": 0.58, "other": 0.65},
}

total_correct = {mid: 0 for mid in accuracy_rates}
total_count = {mid: 0 for mid in accuracy_rates}

for sample in dataset.samples:
    correct_answer = sample.ground_truth
    wrong_answers = [a for a in answer_choices if a != correct_answer]

    for model_id, rates in accuracy_rates.items():
        acc = rates[sample.category]

        # Model responds with the correct letter or a random wrong one
        if random.random() < acc:
            response_text = f"The answer is {correct_answer}"
        else:
            response_text = f"The answer is {random.choice(wrong_answers)}"

        # Compute REAL loss using mmlu_match metric
        loss = mmlu_match(response_text, correct_answer)

        cache.add(
            prompt=sample.prompt,
            model_id=model_id,
            response_text=response_text,
            loss=loss,
        )

        total_count[model_id] += 1
        if loss == 0.0:
            total_correct[model_id] += 1

print(f"Dataset: {len(dataset)} MMLU-style questions")
print(f"Cache: {len(cache)} entries ({len(cache) // len(dataset)} models x {len(dataset)} prompts)")
print(f"\nActual accuracy (from mmlu_match metric):")
for mid in accuracy_rates:
    acc = total_correct[mid] / total_count[mid]
    print(f"  {mid:20s}: {acc:.1%} ({total_correct[mid]}/{total_count[mid]} correct)")

# Show a real cache entry
sample_prompt = dataset.samples[0].prompt
for mid in ["gpt-4o", "llama-3.1-8b"]:
    entry = cache.get(sample_prompt, mid)
    print(f"\nCache entry [{mid}]:")
    print(f"  Response: '{entry.response_text}'")
    print(f"  Ground truth: '{dataset.samples[0].ground_truth}'")
    print(f"  Loss: {entry.loss} ({'CORRECT' if entry.loss == 0.0 else 'WRONG'})")

## 3. Evaluate Baselines

Compare four baseline strategies. Any good router must beat Random and approach Oracle.

In [ ]:
prompt_hashes = [lr.ResponseCache.hash_prompt(s.prompt) for s in dataset.samples]

# Evaluate all 4 baselines
strong_bl = lr.AlwaysStrongBaseline(profiles)
weak_bl = lr.AlwaysWeakBaseline(profiles)
random_bl = lr.RandomBaseline(profiles, seed=42)
oracle_bl = lr.OracleBaseline(profiles)

baselines = {
    "Always Strong": strong_bl.evaluate(cache, prompt_hashes),
    "Always Weak": weak_bl.evaluate(cache, prompt_hashes),
    "Random": random_bl.evaluate(cache, prompt_hashes),
    "Oracle": oracle_bl.evaluate(cache, prompt_hashes),
}

print(f"{'Baseline':<20} {'Accuracy':>10} {'Avg Cost':>12} {'Model':>20}")
print("-" * 65)
for name, results in baselines.items():
    accuracy = 1.0 - sum(r.loss for r in results) / len(results)
    avg_cost = sum(r.cost_per_1k_tokens for r in results) / len(results)
    models_used = set(r.selected_model for r in results)
    print(f"{name:<20} {accuracy:>9.1%} ${avg_cost:>10.6f}   {', '.join(models_used)}")

## 4. Full Router Evaluation with Pareto Sweep

Create a `UniRouteRouter` and run the full evaluator — sweeps lambda from 0 to 50, producing a Pareto curve of cost vs quality tradeoffs.

In [ ]:
# Create the router (lambda=0 means quality-only by default)
router = lr.UniRouteRouter(
    embedder=embedder,
    cluster_assigner=assigner,
    registry=registry,
    cost_weight=0.0,
    use_soft_assignment=False,
)

# Run full evaluation with 25 lambda steps
evaluator = lr.RouterEvaluator(
    router=router,
    cache=cache,
    profiles=profiles,
    lambda_range=(0.0, 50.0),
    lambda_steps=25,
)

result = evaluator.evaluate(dataset, dataset_name="synthetic-bench")
print(result.summary())

## 5. Routing Metrics Deep Dive

All research-standard metrics at a glance.

In [ ]:
m = result.metrics

print("=== Research-Standard Routing Metrics ===\n")
print(f"AUROC:              {m.auroc:.4f}   (1.0 = perfect, 0.5 = random)")
print(f"APGR:               {m.apgr:.2%}   (fraction of strong-weak gap recovered)")
print(f"Win Rate:           {m.win_rate:.2%}   (fraction of prompts answered correctly)")
print()
print("Cost at Performance Target (CPT):")
print(f"  CPT(50%):         {m.cpt_50 or 'N/A'}")
print(f"  CPT(75%):         {m.cpt_75 or 'N/A'}")
print(f"  CPT(90%):         {m.cpt_90 or 'N/A'}")
print(f"  CPT(95%):         {m.cpt_95 or 'N/A'}")
print()
print("Performance Gap Recovered at savings levels:")
print(f"  PGR @ 25% savings: {m.pgr_at_25_savings or 'N/A'}")
print(f"  PGR @ 50% savings: {m.pgr_at_50_savings or 'N/A'}")
print(f"  PGR @ 75% savings: {m.pgr_at_75_savings or 'N/A'}")
print()
print(f"Strong model: {m.strong_model} (accuracy: {m.quality_strong:.1%})")
print(f"Weak model:   {m.weak_model} (accuracy: {m.quality_weak:.1%})")
print(f"Samples:      {m.num_samples}")

## 6. Pareto Curve — Cost vs Quality Tradeoff

As lambda increases, the router shifts from quality-optimized (expensive) to cost-optimized (cheap). The curve shows how quality degrades as we reduce cost.

In [ ]:
print(f"{'Lambda':>8} {'Quality':>10} {'Avg Cost':>12} {'Strong %':>10} {'Model Distribution'}")
print("-" * 85)
for p in result.pareto_curve:
    dist = ", ".join(f"{m}:{pct:.0%}" for m, pct in sorted(p.model_distribution.items()) if pct > 0.01)
    print(f"{p.lambda_value:>8.1f} {p.quality:>9.1%} ${p.avg_cost:>10.6f} {p.strong_model_fraction:>9.1%}   {dist}")

## 7. Visualize Pareto Curve

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Quality vs Cost
    ax1 = axes[0]
    costs = [p.avg_cost for p in result.pareto_curve]
    qualities = [p.quality for p in result.pareto_curve]
    ax1.plot(costs, qualities, "b-o", markersize=5, label="Router (UniRoute)")

    # Plot baselines
    bl_markers = {"always_strong": ("^", "red"), "always_weak": ("v", "green"),
                  "random": ("s", "orange"), "oracle": ("*", "purple")}
    for name, (marker, color) in bl_markers.items():
        ax1.scatter(result.baseline_cost[name], result.baseline_quality[name],
                   marker=marker, color=color, s=120, zorder=5, label=name.replace("_", " ").title())

    ax1.set_xlabel("Average Cost per 1K Tokens ($)")
    ax1.set_ylabel("Quality (Accuracy)")
    ax1.set_title("Cost-Quality Pareto Curve")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Plot 2: Quality vs Strong Model Fraction
    ax2 = axes[1]
    strong_fracs = [p.strong_model_fraction for p in result.pareto_curve]
    ax2.plot(strong_fracs, qualities, "r-o", markersize=5, label="Router")

    # Random baseline line
    ax2.plot([0, 1], [result.baseline_quality["always_weak"],
                      result.baseline_quality["always_strong"]], "k--", alpha=0.5, label="Random (linear)")

    ax2.set_xlabel("Strong Model Call Fraction")
    ax2.set_ylabel("Quality (Accuracy)")
    ax2.set_title("Quality vs Strong Model Usage")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not installed — skipping visualization")
    print("Install with: pip install matplotlib")

## 8. AUROC — Individual Metric Test

The AUROC measures how well the router can distinguish prompts that need the strong model from those that don't.

In [ ]:
# Compute AUROC directly from our cached data
scores = []
labels = []

strong_id = strong_bl.model_id
weak_id = weak_bl.model_id

for sample in dataset.samples:
    strong_entry = cache.get(sample.prompt, strong_id)
    weak_entry = cache.get(sample.prompt, weak_id)
    if strong_entry and weak_entry:
        # Score: how much worse the weak model is (higher = more likely to need strong)
        score = weak_entry.loss - strong_entry.loss
        # Label: strong model was actually needed
        label = weak_entry.loss > 0.0 and strong_entry.loss == 0.0
        scores.append(score)
        labels.append(label)

auroc = lr.compute_auroc(scores, labels)
print(f"AUROC: {auroc:.4f}")
print(f"  Perfect router: 1.0000")
print(f"  Random router:  0.5000")
print()

n_pos = sum(labels)
n_neg = len(labels) - n_pos
print(f"Samples where strong model was needed: {n_pos}/{len(labels)} ({n_pos/len(labels):.1%})")
print(f"Samples where weak model sufficed:     {n_neg}/{len(labels)} ({n_neg/len(labels):.1%})")

## 9. Model Distribution per Lambda

See how the router shifts between models as cost pressure increases.

In [ ]:
try:
    import matplotlib.pyplot as plt

    model_ids = sorted(profiles[0].model_id for _ in [0])  # get all
    model_ids = sorted({m for p in result.pareto_curve for m in p.model_distribution})

    fig, ax = plt.subplots(figsize=(12, 5))
    lambdas = [p.lambda_value for p in result.pareto_curve]

    bottom = np.zeros(len(lambdas))
    colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

    for i, mid in enumerate(model_ids):
        fracs = [p.model_distribution.get(mid, 0.0) for p in result.pareto_curve]
        ax.bar(lambdas, fracs, bottom=bottom, width=1.5, label=mid,
               color=colors[i % len(colors)], alpha=0.85)
        bottom += np.array(fracs)

    ax.set_xlabel("Lambda (cost weight)")
    ax.set_ylabel("Fraction of Prompts")
    ax.set_title("Model Selection Distribution vs Lambda")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlim(-1, max(lambdas) + 1)
    ax.grid(True, alpha=0.2, axis="y")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not installed — skipping visualization")

## 10. Save & Reload Evaluation Results

Results are JSON-serializable for comparison across experiments.

In [ ]:
import json, tempfile, os

# Save to temp file
with tempfile.NamedTemporaryFile(suffix=".json", delete=False, mode="w") as f:
    tmp_path = f.name
    result.save(tmp_path)

# Reload and inspect
with open(tmp_path) as f:
    data = json.load(f)

print("Saved evaluation result keys:", list(data.keys()))
print(f"Pareto curve points: {len(data['pareto_curve'])}")
print(f"Metrics keys: {list(data['metrics'].keys())}")
print(f"\nSample pareto point:")
print(json.dumps(data['pareto_curve'][0], indent=2))

os.unlink(tmp_path)

## 11. Response Cache — Save & Reload

The response cache persists model outputs to JSONL. Once populated, evaluation is instant (no LLM calls).

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False, mode="w") as f:
    cache_path = f.name

# Save the cache
cache.save(cache_path)
file_size = os.path.getsize(cache_path)
print(f"Cache saved: {cache_path}")
print(f"File size: {file_size:,} bytes ({len(cache)} entries)")

# Load into a fresh cache
cache2 = lr.ResponseCache(cache_path)
print(f"\nReloaded cache: {len(cache2)} entries")
print(f"Models: {sorted(cache2.model_ids)}")

# Verify a sample entry
sample_prompt = dataset.samples[0].prompt
entry = cache2.get(sample_prompt, "gpt-4o")
print(f"\nSample entry for '{sample_prompt[:50]}...':")
print(f"  Model: {entry.model_id}")
print(f"  Loss: {entry.loss}")
print(f"  Response: {entry.response_text}")

os.unlink(cache_path)

## 12. Compare Two Routers

Evaluate the same router with soft vs hard cluster assignment to compare approaches.

In [ ]:
# Router A: Hard assignment (one-hot)
router_hard = lr.UniRouteRouter(
    embedder=embedder, cluster_assigner=assigner, registry=registry,
    cost_weight=0.0, use_soft_assignment=False,
)

# Router B: Soft assignment (full probability distribution)
router_soft = lr.UniRouteRouter(
    embedder=embedder, cluster_assigner=assigner, registry=registry,
    cost_weight=0.0, use_soft_assignment=True,
)

results = {}
for name, r in [("Hard Assignment", router_hard), ("Soft Assignment", router_soft)]:
    ev = lr.RouterEvaluator(router=r, cache=cache, profiles=profiles, lambda_steps=15)
    results[name] = ev.evaluate(dataset, dataset_name=name)

print(f"{'Router':<20} {'AUROC':>8} {'APGR':>8} {'Win Rate':>10}")
print("-" * 50)
for name, res in results.items():
    m = res.metrics
    print(f"{name:<20} {m.auroc:>7.4f} {m.apgr:>7.2%} {m.win_rate:>9.2%}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))

    colors = {"Hard Assignment": "#e74c3c", "Soft Assignment": "#3498db"}
    for name, res in results.items():
        costs = [p.avg_cost for p in res.pareto_curve]
        quals = [p.quality for p in res.pareto_curve]
        ax.plot(costs, quals, "-o", markersize=4, color=colors[name], label=name)

    # Add baseline markers
    for bl_name, (marker, color) in bl_markers.items():
        ax.scatter(result.baseline_cost[bl_name], result.baseline_quality[bl_name],
                  marker=marker, color=color, s=100, zorder=5,
                  label=bl_name.replace("_", " ").title())

    ax.set_xlabel("Average Cost per 1K Tokens ($)")
    ax.set_ylabel("Quality (Accuracy)")
    ax.set_title("Hard vs Soft Assignment — Pareto Comparison")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not installed — skipping")

## 13. Unit Tests — Verify SDK Integration

Run the evaluation test suite inline to confirm everything works.

In [ ]:
import subprocess, sys
result_test = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_evaluation.py", "-v", "--tb=short"],
    capture_output=True, text=True, cwd="/Users/diogovieira/Developer/lunar-router"
)
print(result_test.stdout)
if result_test.returncode != 0:
    print(result_test.stderr)

---

# Auto-Training: Self-Improving Router

The sections below demonstrate the full auto-training pipeline:
1. **Preference pairs** from cached benchmark results
2. **Production feedback loop** with simulated traces
3. **Drift detection** for cluster health
4. **AutoTrainer** — the full evaluate-augment-retrain-evaluate-promote cycle

## 14. Generate Preference Pairs from Cache

Extract "model A beat model B" pairs from our cached benchmark results. These train the router to prefer winners on similar prompts.

In [ ]:
# Generate preference pairs from the eval cache
pref_dataset = lr.PreferenceDataset(name="from_cache")
count = pref_dataset.add_from_cache(cache)

print(f"Generated {count} preference pairs from {len(cache)} cached responses")
print(f"\nModel win rates:")
for model, rate in sorted(pref_dataset.model_win_rates().items(), key=lambda x: -x[1]):
    print(f"  {model:20s}: {rate:.1%}")

# Filter by confidence
high_conf = pref_dataset.filter_by_confidence(0.8)
print(f"\nHigh-confidence pairs: {len(high_conf)}/{len(pref_dataset)}")

## 15. Production Feedback Loop — Simulated Traces

Simulate 1,000 production traces with error patterns. The `TraceToTraining` module converts these into production Psi vectors and blends them with the benchmark profiles.

In [ ]:
from lunar_router.feedback.trace_to_training import TraceToTraining, TraceRecord

random.seed(123)

# Simulate 1000 production traces across 5 clusters and 4 models
traces = []
model_ids = [p.model_id for p in profiles]

for i in range(1000):
    mid = random.choice(model_ids)
    cluster = random.randint(0, 4)

    # Simulate errors: use psi_vector rates but with production noise
    profile = next(p for p in profiles if p.model_id == mid)
    base_error = profile.psi_vector[cluster]
    # Production has 20% higher error rates (model degradation)
    production_error = min(base_error * 1.2, 1.0)
    is_error = random.random() < production_error

    traces.append(TraceRecord(
        request_id=f"req_{i}",
        selected_model=mid,
        cluster_id=cluster,
        is_error=is_error,
        latency_ms=random.uniform(100, 5000),
        total_cost_usd=profile.cost_per_1k_tokens * random.uniform(0.5, 2.0),
    ))

print(f"Simulated {len(traces)} production traces")

# Convert to training signal
converter = TraceToTraining(num_clusters=5)
converter.add_traces(traces)

# Compute production Psi vectors
updates = converter.compute_psi_updates()
print(f"\nProduction Psi updates for {len(updates)} models:")
for u in updates:
    print(f"  {u.model_id:20s}: error={u.error_rate:.1%}, traces={u.total_traces}, avg_latency={u.avg_latency_ms:.0f}ms")

# Blend production + benchmark (alpha=0.3 = 30% production, 70% benchmark)
blended_profiles = converter.blend_with_profiles(profiles, alpha=0.3)

print(f"\nError rate changes after blending (alpha=0.3):")
for old, new in zip(profiles, blended_profiles):
    delta = new.overall_error_rate - old.overall_error_rate
    print(f"  {old.model_id:20s}: {old.overall_error_rate:.3f} -> {new.overall_error_rate:.3f} ({delta:+.3f})")

## 16. Drift Detection

Check if production prompt embeddings are drifting away from trained cluster centroids. If drift exceeds threshold, re-clustering is needed.

In [ ]:
# Simulate embeddings — some near centroids, some drifted
normal_embeddings = np.random.randn(100, 2) * 0.3 + centroids[np.random.randint(0, 5, 100)]
drifted_embeddings = np.random.randn(100, 2) * 0.3 + np.array([3.0, 3.0])  # far from all centroids

# Check normal traffic (no drift expected)
detector = lr.DriftDetector(assigner, drift_threshold=1.5)
normal_report = detector.check(normal_embeddings)
print("=== Normal Traffic ===")
print(normal_report.summary())

# Check drifted traffic
mixed = np.vstack([normal_embeddings[:50], drifted_embeddings[:50]])
np.random.shuffle(mixed)
drifted_report = detector.check(mixed)
print(f"\n=== Mixed Traffic (50% drifted) ===")
print(drifted_report.summary())

## 17. AutoTrainer — Full Self-Improving Cycle

The `AutoTrainer` chains the entire improvement loop:

**evaluate baseline -> generate preferences -> refine Psi vectors -> evaluate new -> quality gate -> promote or reject**

No LLM calls needed here — works entirely from the response cache.

In [ ]:
# Create the AutoTrainer
trainer = lr.AutoTrainer(
    embedder=embedder,
    cluster_assigner=assigner,
    profiles=profiles,
    eval_dataset=dataset,
    eval_cache=cache,
    cost_weight=0.0,
    config=lr.AutoTrainConfig(
        use_judge=False,  # no real LLM judge in this demo
        production_alpha=0.3,
        min_auroc_improvement=0.0,  # accept any non-regression
        min_win_rate=0.5,
        max_error_rate_increase=0.1,
        lambda_steps=15,
    ),
)

# Run auto-training from cache only (no LLM calls)
print("Running auto-training from cache...")
auto_result = trainer.train_from_cache()
print(auto_result.summary())

## 18. AutoTrainer with Production Traces

Feed production traces into the auto-trainer. It blends production error rates with benchmark profiles and promotes only if quality gates pass.

In [ ]:
# Run auto-training with production traces
print("Running auto-training with production traces...")
trace_result = trainer.train_from_traces(traces)
print(trace_result.summary())

# Show training history
print(f"\n=== Training History ({len(trainer.history)} cycles) ===")
for i, r in enumerate(trainer.history):
    status = "PROMOTED" if r.promoted else "REJECTED"
    auroc = r.new_metrics.auroc if r.new_metrics else 0
    print(f"  Cycle {i+1}: {status} | AUROC={auroc:.4f} | pairs={r.preference_pairs_generated} | traces={r.production_traces_used}")

## 19. Get the Improved Router

After auto-training, get the promoted router with updated Psi vectors.

In [ ]:
# Get the improved router
improved_router = trainer.get_router()

# Compare routing decisions: original vs improved
test_prompts = [
    "[math] What is the integral of x^2?",
    "[creative] Write a haiku about rain",
    "[coding] Implement binary search in Python",
]

print(f"{'Prompt':<50} {'Original':>20} {'Improved':>20}")
print("-" * 92)
for prompt in test_prompts:
    original_decision = router.route(prompt)
    improved_decision = improved_router.route(prompt)
    print(f"{prompt[:48]:48s}   {original_decision.selected_model:>18s}   {improved_decision.selected_model:>18s}")

# Show Psi vector changes
print("\n=== Psi Vector Changes ===")
for old_p, new_p in zip(profiles, trainer.profiles):
    delta = new_p.psi_vector - old_p.psi_vector
    max_change = np.abs(delta).max()
    print(f"  {old_p.model_id:20s}: max |delta|={max_change:.4f}, accuracy {old_p.overall_accuracy:.2%} -> {new_p.overall_accuracy:.2%}")